# Lesson 2.4 - Applied Stats for ML (Iris classification)

## Objectives
- Train a simple classifier on Iris data.
- Evaluate multiple metrics beyond accuracy.
- Interpret metric trade-offs for business decisions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)

IRIS_URL = 'https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv'
iris_df = pd.read_csv(IRIS_URL)
print('Loaded shape:', iris_df.shape)
print(iris_df.head())

## Section A - Inspect & Prepare Data

In [ ]:
print(iris_df.describe(include='all'))

feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[feature_cols]
y = iris_df['species']

print('Classes:', y.unique())

## Section B - Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train_scaled.shape)
print('Test shape:', X_test_scaled.shape)

## Section C - Train a Simple Classifier

In [ ]:
clf = LogisticRegression(max_iter=300)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print('Predictions sample:', y_pred[:10])

## Section D - Evaluate with Multiple Metrics

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred)

print('Accuracy:', acc)
print('Precision (macro):', prec)
print('Recall (macro):', rec)
print('F1 (macro):', f1)
print('Confusion matrix:
', cm)

# One-vs-rest AUC
classes = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)
y_proba = clf.predict_proba(X_test_scaled)
auc_macro = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
print('ROC-AUC (macro, OVR):', auc_macro)

## Section E - Interpret Metrics

Accuracy gives overall correctness, while precision and recall expose class-specific trade-offs. F1 balances precision/recall. In risk-sensitive applications, precision may matter more for false-positive costs, while recall may matter more when missing positives is costly.

## Section F - Overfitting/Underfitting Demo (Simple)

In [ ]:
# Too-simple baseline: use one feature only
X_one = iris_df[['petal_length']]
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X_one, y, test_size=0.25, random_state=42, stratify=y
)

scaler_1 = StandardScaler()
X1_train_s = scaler_1.fit_transform(X1_train)
X1_test_s = scaler_1.transform(X1_test)

clf_simple = LogisticRegression(max_iter=300)
clf_simple.fit(X1_train_s, y1_train)
y1_pred = clf_simple.predict(X1_test_s)

print('Simple-model accuracy:', accuracy_score(y1_test, y1_pred))
print('Simple-model F1 (macro):', f1_score(y1_test, y1_pred, average='macro'))

## Business Logic & Exceptions

Choosing the wrong metric can lead to poor business outcomes even when accuracy looks good. Production monitoring should track multiple metrics, drift signals, and threshold-based alerts.

In [ ]:
def assert_minimum_performance(acc, threshold=0.8):
    if acc < threshold:
        raise ValueError(f'Model accuracy {acc:.2f} below required threshold {threshold}')

assert_minimum_performance(acc, threshold=0.7)
print('Performance check passed')

## Interview Questions & Answers

1. **Q: Accuracy vs precision vs recall?**  
   **A:** Accuracy is global correctness; precision penalizes false positives; recall penalizes false negatives.

2. **Q: What does F1 score capture?**  
   **A:** Harmonic balance between precision and recall.

3. **Q: Why not use accuracy alone?**  
   **A:** It can hide poor minority-class performance.

4. **Q: What is ROC-AUC at high level?**  
   **A:** Ranking quality across thresholds (especially in binary/OVR framing).

5. **Q: Example where recall matters most?**  
   **A:** Medical screening where missed positives are costly.

6. **Q: Example where precision matters most?**  
   **A:** Fraud lockouts where false alarms hurt user trust.

7. **Q: Bias vs variance trade-off?**  
   **A:** Higher bias underfits; higher variance overfits.

8. **Q: Why scale features for logistic regression?**  
   **A:** Scaling stabilizes optimization and makes coefficients more comparable.

9. **Q: What does confusion matrix show?**  
   **A:** Per-class prediction errors and correct counts.

10. **Q: Why monitor metrics in production?**  
    **A:** Data drift can degrade real-world performance after deployment.